In [ ]:
# baixar e instalar o node
!curl -fsSL https://deb.nodesource.com/setup_22.x -o /tmp/nodesource.sh
!bash /tmp/nodesource.sh
!apt-get install -y nodejs
!node --version
!npm --version

In [ ]:
# instalar opencode e rea
!npm install -g opencode-ai rea-agents
!opencode --version
!rea --help

In [ ]:
# instalar Ghidra
import subprocess
import os
import sys
import json
import urllib.request
import zipfile
import shutil
import glob

# 1. Instalar dependências
subprocess.run(
    ["bash", "-lc",
     "apt-get update -qq && apt-get install -y openjdk-21-jdk unzip"],
    check=True
)

# 2. Configurar Java
java = shutil.which("java")
if not java:
    raise RuntimeError("Java não foi encontrado.")

java_real = os.path.realpath(java)
java_home = os.path.dirname(os.path.dirname(java_real))

os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = java_home + "/bin:" + os.environ["PATH"]

subprocess.run(["java", "-version"], check=True)

# 3. Consultar o último lançamento oficial do Ghidra
api = "https://api.github.com/repos/NationalSecurityAgency/ghidra/releases/latest"

req = urllib.request.Request(
    api,
    headers={"User-Agent": "Google-Colab-Ghidra-Installer"}
)

with urllib.request.urlopen(req, timeout=30) as response:
    release = json.load(response)

assets = [
    a for a in release["assets"]
    if a["name"].endswith(".zip")
    and a["name"].startswith("ghidra_")
    and "_PUBLIC_" in a["name"]
]

if not assets:
    raise RuntimeError("Não encontrei o ZIP oficial do Ghidra.")

asset = assets[0]
zip_path = "/content/" + asset["name"]

print("Versão:", release["tag_name"])
print("Baixando:", asset["name"])

urllib.request.urlretrieve(asset["browser_download_url"], zip_path)

# 4. Extrair
with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content")

ghidra_dirs = glob.glob("/content/ghidra_*_PUBLIC")

if not ghidra_dirs:
    raise RuntimeError("Não encontrei a pasta extraída do Ghidra.")

ghidra_dir = ghidra_dirs[0]
os.environ["GHIDRA_INSTALL_DIR"] = ghidra_dir

print("\nGhidra instalado em:", ghidra_dir)
print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("GHIDRA_INSTALL_DIR:", os.environ["GHIDRA_INSTALL_DIR"])

In [4]:
# registrar ghidra no rea
import os
import subprocess
import shutil

env = os.environ.copy()

print("GHIDRA_INSTALL_DIR =", env["GHIDRA_INSTALL_DIR"])
print("JAVA_HOME =", env["JAVA_HOME"])

rea = shutil.which("rea")

if rea:
    subprocess.run([rea, "doctor", "--json"], env=env)
else:
    subprocess.run(["npx", "-y", "rea-agents@latest",
                    "doctor", "--json"], env=env)

GHIDRA_INSTALL_DIR = /content/ghidra_12.1.4_PUBLIC
JAVA_HOME = /usr/lib/jvm/java-21-openjdk-amd64


In [5]:
import os, json

os.makedirs("/content/projeto", exist_ok=True)
os.chdir("/content/projeto")

config = {
    "$schema": "https://opencode.ai/config.json",
    "mcp": {
        "rea": {
            "type": "local",
            "command": ["npx", "-y", "rea-agents@latest", "mcp"],
            "enabled": True,
            "timeout": 30000
        }
    }
}

with open("opencode.json", "w") as f:
    json.dump(config, f, indent=2)

print("Configuração criada em /content/projeto/opencode.json")

Configuração criada em /content/projeto/opencode.json


In [ ]:
# upload de programa ou projeto
import os
import zipfile
import shutil
from google.colab import files

# Pastas do projeto
PROJETO = "/content/projeto"
PASTA_TRABALHO = os.path.join(PROJETO, "arquivos")

os.makedirs(PROJETO, exist_ok=True)
os.makedirs(PASTA_TRABALHO, exist_ok=True)

# Selecionar e enviar o ZIP do PC
print("Selecione o arquivo ZIP do seu computador:")
enviados = files.upload()

if not enviados:
    raise RuntimeError("Nenhum arquivo foi enviado.")

# Pegar o ZIP enviado
nome_zip = next(iter(enviados))
caminho_zip = os.path.join(PROJETO, nome_zip)

# Mover o arquivo enviado para a pasta do projeto
if os.path.abspath(nome_zip) != os.path.abspath(caminho_zip):
    shutil.move(nome_zip, caminho_zip)

# Extrair com validação para evitar caminhos fora da pasta de destino
with zipfile.ZipFile(caminho_zip, "r") as zip_ref:
    destino = os.path.realpath(PASTA_TRABALHO)

    for membro in zip_ref.infolist():
        caminho = os.path.realpath(
            os.path.join(destino, membro.filename)
        )
        if os.path.commonpath([destino, caminho]) != destino:
            raise ValueError(
                f"ZIP contém caminho inseguro: {membro.filename}"
            )

    zip_ref.extractall(PASTA_TRABALHO)

print("\nZIP enviado:", caminho_zip)
print("Arquivos extraídos em:", PASTA_TRABALHO)

print("\nConteúdo extraído:")
for raiz, pastas, arquivos_lista in os.walk(PASTA_TRABALHO):
    nivel = raiz.replace(PASTA_TRABALHO, "").count(os.sep)
    if nivel > 2:
        pastas[:] = []
        continue

    print("  " * nivel + os.path.basename(raiz) + "/")
    for nome in arquivos_lista[:15]:
        print("  " * (nivel + 1) + nome)

In [7]:
#checar arquivos upados
import os

pasta = "/content/projeto/arquivos"

print("Existe:", os.path.isdir(pasta))
print("Conteúdo:")
for item in os.listdir(pasta):
    print(item)

Existe: True
Conteúdo:
jellyfin


In [8]:
# iniciar open code na pasta do sound forge
import subprocess
import time
from google.colab import output

projeto = "/content/projeto/arquivos/"

# Verificar a pasta
import os
assert os.path.isdir(projeto), f"Pasta não encontrada: {projeto}"

# Encerrar o servidor anterior, se estiver ativo
try:
    proc.terminate()
    proc.wait(timeout=5)
except Exception:
    pass

# Iniciar a partir da pasta do projeto
proc = subprocess.Popen(
    [
        "opencode", "web",
        "--hostname", "0.0.0.0",
        "--port", "4096",
        "--print-logs"
    ],
    cwd=projeto,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

time.sleep(4)

if proc.poll() is None:
    print("OpenCode iniciado na pasta:", projeto)
    output.serve_kernel_port_as_window(4096)
else:
    print("O OpenCode encerrou. Código:", proc.returncode)

OpenCode iniciado na pasta: /content/projeto/arquivos/
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
# baixar cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!/usr/local/bin/cloudflared --version

In [10]:
# gerar link de acesso
import subprocess
import time
import re

cmd = [
    "/usr/local/bin/cloudflared",
    "tunnel",
    "--url", "http://127.0.0.1:4096",
    "--no-autoupdate"
]

tunnel = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for _ in range(60):
    line = tunnel.stdout.readline()
    if not line:
        if tunnel.poll() is not None:
            print("O cloudflared encerrou.")
            break
        continue

    print(line, end="")

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        line
    )
    if match:
        print("\nURL PARA ABRIR NO SEU PC:")
        print(match.group(0))
        break

2026-10-18T23:18:54Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-18T23:18:54Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-18T23:18:57Z INF +--------------------------------------------------------------------------------------------+
2026-10-18T23:18:57Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-18T23:18:57Z INF |  https://dosage-environments-scuba-cgi.trycloudflare.c

In [ ]:
# manter sessão ativa
import time
while True:
  time.sleep(3)

